In [0]:
# ==============================================================================
# TASK 7.1: DATE DIMENSION GENERATION (2016-2020)
# ==============================================================================

from pyspark.sql import functions as F

# 1. GENERATE DAILY DATE SEQUENCE (2016 TO 2020)
df_dates = spark.sql("""
    SELECT explode(sequence(to_date('2016-01-01'), to_date('2020-12-31'), interval 1 day)) as full_date
""")

# 2. ENRICH WITH CALENDAR, DESCRIPTIVE, FLAG, AND FISCAL FIELDS
dim_date_df = (
    df_dates
    # Primary Key
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    
    # Standard Calendar Fields
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))  # 1=Sun, 7=Sat
    .withColumn("week_of_year", F.weekofyear("full_date"))
    
    # Descriptive Names
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("month_name_short", F.date_format("full_date", "MMM"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("day_name_short", F.date_format("full_date", "E"))
    .withColumn("year_month", F.date_format("full_date", "yyyy-MM"))
    
    # Operational Flags
    .withColumn("is_weekend", F.when(F.dayofweek("full_date").isin(1, 7), True).otherwise(False))
    .withColumn("is_month_start", F.col("full_date") == F.trunc("full_date", "MM"))
    .withColumn("is_month_end", F.col("full_date") == F.last_day("full_date"))
    
    # Fiscal Calendar Fields (Fiscal Year starts April 1)
    .withColumn("fiscal_year", 
        F.when(F.month("full_date") >= 4, F.year("full_date"))
         .otherwise(F.year("full_date") - 1)
    )
    .withColumn("fiscal_quarter", 
        F.when(F.month("full_date") >= 4, F.ceil((F.month("full_date") - 3) / 3))
         .otherwise(F.ceil((F.month("full_date") + 9) / 3))
    )
    .withColumn("fiscal_month", 
        F.when(F.month("full_date") >= 4, F.month("full_date") - 3)
         .otherwise(F.month("full_date") + 9)
    )
)

# 3. WRITE TO DELTA TABLE
target_table = "globalmart.silver.dim_date"

dim_date_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_table)

print(f"✅ Populated `{target_table}` with {spark.table(target_table).count():,} records.")

In [0]:
%sql
-- Verification Query: Date Dimension Validation
SELECT 
    date_key,
    full_date,
    year,
    quarter,
    month_name_short,
    day_name_short,
    is_weekend,
    is_month_start,
    is_month_end,
    fiscal_year,
    fiscal_quarter,
    fiscal_month
FROM globalmart.silver.dim_date
WHERE full_date IN ('2016-01-01', '2016-03-31', '2016-04-01', '2020-12-31')
ORDER BY full_date;

In [0]:
# ==============================================================================
# TASK 7.2: SURROGATE KEY STRATEGY TESTING
# ==============================================================================

from pyspark.sql import functions as F

source_df = spark.table("globalmart.bronze.bronze_customers")

# ------------------------------------------------------------------------------
# RUN 1: Generate Monotonic IDs vs Deterministic Hashes
# ------------------------------------------------------------------------------
run1_df = (
    source_df
    .withColumn("sk_monotonic_run1", F.monotonically_increasing_id())
    .withColumn("sk_hash_run1", F.sha2(F.concat_ws("||", F.col("customer_id")), 256))
    .select("customer_id", "sk_monotonic_run1", "sk_hash_run1")
)

# ------------------------------------------------------------------------------
# RUN 2: Re-run identical transformations on the same table
# ------------------------------------------------------------------------------
run2_df = (
    source_df
    .withColumn("sk_monotonic_run2", F.monotonically_increasing_id())
    .withColumn("sk_hash_run2", F.sha2(F.concat_ws("||", F.col("customer_id")), 256))
    .select("customer_id", "sk_monotonic_run2", "sk_hash_run2")
)

# ------------------------------------------------------------------------------
# COMPARISON: Join Run 1 and Run 2 to check key stability
# ------------------------------------------------------------------------------
comparison_df = (
    run1_df.alias("r1")
    .join(run2_df.alias("r2"), "customer_id")
    .withColumn("monotonic_matches", F.col("sk_monotonic_run1") == F.col("sk_monotonic_run2"))
    .withColumn("hash_matches", F.col("sk_hash_run1") == F.col("sk_hash_run2"))
)

# Check stability summary counts
summary_df = comparison_df.agg(
    F.count("customer_id").alias("total_rows"),
    F.sum(F.when(F.col("monotonic_matches"), 1).otherwise(0)).alias("monotonic_stable_count"),
    F.sum(F.when(F.col("hash_matches"), 1).otherwise(0)).alias("hash_stable_count")
)

display(summary_df)

In [0]:
%sql
-- Sample output comparing Run 1 and Run 2 surrogate keys
SELECT 
    customer_id,
    sk_monotonic_run1,
    sk_monotonic_run2,
    (sk_monotonic_run1 = sk_monotonic_run2) AS is_monotonic_stable,
    SUBSTRING(sk_hash_run1, 1, 12) AS sk_hash_run1_sample,
    SUBSTRING(sk_hash_run2, 1, 12) AS sk_hash_run2_sample,
    (sk_hash_run1 = sk_hash_run2) AS is_hash_stable
FROM (
    SELECT 
        customer_id,
        monotonically_increasing_id() AS sk_monotonic_run1,
        sha2(customer_id, 256) AS sk_hash_run1
    FROM globalmart.bronze.bronze_customers
) r1
JOIN (
    SELECT 
        customer_id,
        monotonically_increasing_id() AS sk_monotonic_run2,
        sha2(customer_id, 256) AS sk_hash_run2
    FROM globalmart.bronze.bronze_customers
) r2 USING (customer_id)
LIMIT 10;

In [0]:
%sql
SHOW TABLES IN globalmart.bronze;

In [0]:
# ==============================================================================
# STEP 1: CONFORMANCE VALIDATION & ERROR LOGGING
# ==============================================================================

from pyspark.sql import functions as F

# 1. Create Metadata Log Table
spark.sql("""
CREATE TABLE IF NOT EXISTS globalmart.silver.conformance_error_log (
    log_id STRING,
    check_name STRING,
    violation_type STRING,
    category_value STRING,
    logged_at TIMESTAMP
) USING DELTA;
""")

# 2. Load Bronze Data using exact table names
df_products = spark.table("globalmart.bronze.bronze_products")
df_translation = spark.table("globalmart.bronze.bronze_product_category_name_translation")

# 3. Check 1: Missing English Translations
missing_english_cats = (
    df_products
    .filter(F.col("product_category_name").isNotNull())
    .join(df_translation, "product_category_name", "left")
    .filter(F.col("product_category_name_english").isNull())
    .select(F.col("product_category_name").alias("category_value"))
    .distinct()
    .withColumn("log_id", F.expr("uuid()"))
    .withColumn("check_name", F.lit("Category Translation Conformance"))
    .withColumn("violation_type", F.lit("Missing English Translation"))
    .withColumn("logged_at", F.current_timestamp())
)

# 4. Check 2: Non 1:1 Mappings (Multiple English translations for one Portuguese category)
duplicate_mappings = (
    df_translation
    .groupBy("product_category_name")
    .agg(F.countDistinct("product_category_name_english").alias("trans_count"))
    .filter(F.col("trans_count") > 1)
    .select(F.col("product_category_name").alias("category_value"))
    .withColumn("log_id", F.expr("uuid()"))
    .withColumn("check_name", F.lit("Category Mapping 1:1 Conformance"))
    .withColumn("violation_type", F.lit("Duplicate English Mappings for Single Category"))
    .withColumn("logged_at", F.current_timestamp())
)

# 5. Union & Log Violations
violations_df = missing_english_cats.unionByName(duplicate_mappings)
violation_count = violations_df.count()

if violation_count > 0:
    violations_df.write.format("delta").mode("append").saveAsTable("globalmart.silver.conformance_error_log")
    print(f"⚠️ Logged {violation_count} conformance violations to `globalmart.silver.conformance_error_log`.")
else:
    print("✅ Conformance Check Passed: No translation violations found.")

In [0]:
# ==============================================================================
# STEP 2: BUILD DIM_PRODUCT (SCD TYPE 1)
# ==============================================================================

# 1. Create Target Delta Dimension Table
spark.sql("""
CREATE TABLE IF NOT EXISTS globalmart.silver.dim_product (
    product_sk STRING,
    product_id STRING,
    product_category_name STRING,
    product_category_name_english STRING,
    product_name_lenght INT,
    product_description_lenght INT,
    product_photos_qty INT,
    product_weight_g INT,
    product_length_cm INT,
    product_height_cm INT,
    product_width_cm INT,
    updated_at TIMESTAMP
) USING DELTA;
""")

# 2. Stage Source Data with SHA-256 Surrogate Key & Fallback for Missing Categories
staging_products = (
    df_products
    .join(df_translation, "product_category_name", "left")
    .withColumn("product_sk", F.sha2(F.concat_ws("||", F.col("product_id")), 256))
    .withColumn("product_category_name_english", 
        F.coalesce(F.col("product_category_name_english"), F.lit("Unknown"))
    )
    .withColumn("updated_at", F.current_timestamp())
    .select(
        "product_sk", "product_id", "product_category_name", 
        "product_category_name_english", "product_name_lenght", 
        "product_description_lenght", "product_photos_qty", 
        "product_weight_g", "product_length_cm", "product_height_cm", 
        "product_width_cm", "updated_at"
    )
)

staging_products.createOrReplaceTempView("stg_dim_product")

# 3. Merge Logic
spark.sql("""
MERGE INTO globalmart.silver.dim_product AS target
USING stg_dim_product AS source
ON target.product_id = source.product_id
WHEN MATCHED AND (
    NVL(target.product_category_name, '') <> NVL(source.product_category_name, '') OR
    NVL(target.product_category_name_english, '') <> NVL(source.product_category_name_english, '') OR
    NVL(target.product_weight_g, 0) <> NVL(source.product_weight_g, 0) OR
    NVL(target.product_length_cm, 0) <> NVL(source.product_length_cm, 0) OR
    NVL(target.product_height_cm, 0) <> NVL(source.product_height_cm, 0) OR
    NVL(target.product_width_cm, 0) <> NVL(source.product_width_cm, 0)
) THEN UPDATE SET
    target.product_category_name = source.product_category_name,
    target.product_category_name_english = source.product_category_name_english,
    target.product_name_lenght = source.product_name_lenght,
    target.product_description_lenght = source.product_description_lenght,
    target.product_photos_qty = source.product_photos_qty,
    target.product_weight_g = source.product_weight_g,
    target.product_length_cm = source.product_length_cm,
    target.product_height_cm = source.product_height_cm,
    target.product_width_cm = source.product_width_cm,
    target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
    product_sk, product_id, product_category_name, product_category_name_english,
    product_name_lenght, product_description_lenght, product_photos_qty,
    product_weight_g, product_length_cm, product_height_cm, product_width_cm, updated_at
) VALUES (
    source.product_sk, source.product_id, source.product_category_name, source.product_category_name_english,
    source.product_name_lenght, source.product_description_lenght, source.product_photos_qty,
    source.product_weight_g, source.product_length_cm, source.product_height_cm, source.product_width_cm, source.updated_at
);
""")

print(f"🎉 Pipeline Executed Successfully! `dim_product` row count: {spark.table('globalmart.silver.dim_product').count():,}")

In [0]:
%sql
-- Check logged violations
SELECT * FROM globalmart.silver.conformance_error_log;

In [0]:
%sql
-- Check the dimension table output
SELECT 
    product_sk,
    product_id,
    product_category_name,
    product_category_name_english,
    updated_at
FROM globalmart.silver.dim_product
LIMIT 5;

In [0]:
# ==============================================================================
# TASK 7.4: SELLER DIMENSION (SCD TYPE 1 PIPELINE)
# ==============================================================================

from pyspark.sql import functions as F

# 1. CREATE TARGET DELTA DIMENSION TABLE
spark.sql("""
CREATE TABLE IF NOT EXISTS globalmart.silver.dim_seller (
    seller_sk STRING,
    seller_id STRING,
    seller_zip_code_prefix INT,
    seller_city STRING,
    seller_state STRING,
    updated_at TIMESTAMP
) USING DELTA;
""")

# 2. LOAD BRONZE SOURCE DATA & STAGE WITH DETERMINISTIC SK (SHA-256)
df_sellers = spark.table("globalmart.bronze.bronze_sellers")

staging_sellers = (
    df_sellers
    .withColumn("seller_sk", F.sha2(F.concat_ws("||", F.col("seller_id")), 256))
    .withColumn("seller_zip_code_prefix", F.col("seller_zip_code_prefix").cast("int"))
    .withColumn("updated_at", F.current_timestamp())
    .select(
        "seller_sk",
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "updated_at"
    )
)

staging_sellers.createOrReplaceTempView("stg_dim_seller")

# 3. EXECUTE MERGE FOR SCD TYPE 1 (OVERWRITE ON CHANGE / INSERT ON NEW)
spark.sql("""
MERGE INTO globalmart.silver.dim_seller AS target
USING stg_dim_seller AS source
ON target.seller_id = source.seller_id
WHEN MATCHED AND (
    NVL(target.seller_zip_code_prefix, 0) <> NVL(source.seller_zip_code_prefix, 0) OR
    NVL(target.seller_city, '') <> NVL(source.seller_city, '') OR
    NVL(target.seller_state, '') <> NVL(source.seller_state, '')
) THEN UPDATE SET
    target.seller_zip_code_prefix = source.seller_zip_code_prefix,
    target.seller_city = source.seller_city,
    target.seller_state = source.seller_state,
    target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
    seller_sk,
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state,
    updated_at
) VALUES (
    source.seller_sk,
    source.seller_id,
    source.seller_zip_code_prefix,
    source.seller_city,
    source.seller_state,
    source.updated_at
);
""")

print(f"✅ `dim_seller` successfully populated with {spark.table('globalmart.silver.dim_seller').count():,} records.")

In [0]:
%sql
-- Verification Query: Inspect dim_seller records
SELECT 
    SUBSTRING(seller_sk, 1, 12) AS seller_sk_sample,
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state,
    updated_at
FROM globalmart.silver.dim_seller
LIMIT 10;

In [0]:
# ==============================================================================
# TASK 7.5: STEP 1 - INITIAL LOAD (SCD TYPE 2 CUSTOMER DIMENSION)
# ==============================================================================

from pyspark.sql import functions as F

# 1. CREATE TARGET DELTA DIMENSION TABLE
spark.sql("""
CREATE TABLE IF NOT EXISTS globalmart.silver.dim_customer (
    customer_sk STRING,
    customer_id STRING,
    customer_unique_id STRING,
    customer_zip_code_prefix INT,
    customer_city STRING,
    customer_state STRING,
    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,
    is_current BOOLEAN,
    version INT
) USING DELTA;
""")

# 2. LOAD BRONZE SOURCE DATA
df_bronze_cust = spark.table("globalmart.bronze.bronze_customers")

# 3. STAGE INITIAL VERSION 1 DATA
initial_start_date = "2016-01-01 00:00:00"

stg_dim_customer_v1 = (
    df_bronze_cust
    .withColumn("effective_start_date", F.to_timestamp(F.lit(initial_start_date)))
    .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn("version", F.lit(1))
    .withColumn("customer_zip_code_prefix", F.col("customer_zip_code_prefix").cast("int"))
    # Deterministic SK: Hash of natural key + effective start date
    .withColumn("customer_sk", F.sha2(
        F.concat_ws("||", F.col("customer_id"), F.date_format("effective_start_date", "yyyy-MM-dd HH:mm:ss")), 256
    ))
    .select(
        "customer_sk", "customer_id", "customer_unique_id",
        "customer_zip_code_prefix", "customer_city", "customer_state",
        "effective_start_date", "effective_end_date", "is_current", "version"
    )
)

# Overwrite for clean initial setup
stg_dim_customer_v1.write.format("delta").mode("overwrite").saveAsTable("globalmart.silver.dim_customer")

print(f"✅ Initial Load Complete! `dim_customer` total records: {spark.table('globalmart.silver.dim_customer').count():,}")

In [0]:
# ==============================================================================
# TASK 7.5: STEP 2 - SIMULATE CHANGE & APPLY SCD TYPE 2 LOGIC
# ==============================================================================

# 1. SELECT 5 SAMPLE CUSTOMERS TO SIMULATE LOCATION UPDATES
sample_customers = (
    spark.table("globalmart.silver.dim_customer")
    .select("customer_id", "customer_unique_id", "version")
    .limit(5)
    .collect()
)

target_customer_ids = [row["customer_id"] for row in sample_customers]
print(f"🎯 Simulating updates for customer_ids: {target_customer_ids}")

# 2. CREATE STAGING UPDATES WITH SIMULATED NEW LOCATIONS
change_timestamp = "2026-07-25 10:00:00"

updates_df = (
    spark.table("globalmart.bronze.bronze_customers")
    .filter(F.col("customer_id").isin(target_customer_ids))
    .withColumn("customer_city", F.concat(F.col("customer_city"), F.lit("_relocated")))
    .withColumn("customer_state", F.lit("SP"))  # Relocate to SP
    .withColumn("effective_start_date", F.to_timestamp(F.lit(change_timestamp)))
    .withColumn("customer_zip_code_prefix", F.col("customer_zip_code_prefix").cast("int"))
)

# 3. COMBINE UNCHANGED RECORD KEYS WITH NEW VERSION RECORDS (2-PASS MERGE PATTERN)
# Pass A: Null key joiner to force insertion of new versions
new_versions_df = updates_df.withColumn("merge_key", F.lit(None).cast("string"))

# Pass B: Real key joiner to match and close out current version 1 records
existing_matches_df = updates_df.withColumn("merge_key", F.col("customer_id"))

staging_scd2_df = existing_matches_df.unionByName(new_versions_df)
staging_scd2_df.createOrReplaceTempView("stg_customer_updates")

# 4. DELTA MERGE FOR SCD TYPE 2
spark.sql("""
MERGE INTO globalmart.silver.dim_customer AS target
USING (
    SELECT 
        merge_key,
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state,
        effective_start_date
    FROM stg_customer_updates
) AS source
ON target.customer_id = source.merge_key AND target.is_current = TRUE
-- Pass 1: Expire old active version
WHEN MATCHED AND (
    target.customer_city <> source.customer_city OR 
    target.customer_state <> source.customer_state
) THEN UPDATE SET
    target.is_current = FALSE,
    target.effective_end_date = source.effective_start_date
-- Pass 2: Insert new version
WHEN NOT MATCHED THEN INSERT (
    customer_sk,
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state,
    effective_start_date,
    effective_end_date,
    is_current,
    version
) VALUES (
    sha2(concat_ws('||', source.customer_id, date_format(source.effective_start_date, 'yyyy-MM-dd HH:mm:ss')), 256),
    source.customer_id,
    source.customer_unique_id,
    source.customer_zip_code_prefix,
    source.customer_city,
    source.customer_state,
    source.effective_start_date,
    NULL,
    TRUE,
    2
);
""")

print("🎉 SCD Type 2 merge completed successfully!")

In [0]:
%sql
-- Deliverable Verification: View both Version 1 (Expired) and Version 2 (Current) for changed customer
SELECT 
    SUBSTRING(customer_sk, 1, 12) AS customer_sk_sample,
    customer_id,
    customer_city,
    customer_state,
    effective_start_date,
    effective_end_date,
    is_current,
    version
FROM globalmart.silver.dim_customer
WHERE customer_id IN (
    SELECT customer_id 
    FROM globalmart.silver.dim_customer 
    GROUP BY customer_id 
    HAVING COUNT(1) > 1
)
ORDER BY customer_id, version;

In [0]:
# ==============================================================================
# TASK 7.6: FACT SALES TABLE (DELIVERED ORDERS ONLY)
# ==============================================================================

from pyspark.sql import functions as F

# 1. CREATE TARGET DELTA FACT TABLE
spark.sql("""
CREATE TABLE IF NOT EXISTS globalmart.silver.fct_sales (
    sales_fact_sk STRING,
    -- Foreign Keys to Dimensions
    customer_sk STRING,
    seller_sk STRING,
    product_sk STRING,
    order_date_key INT,
    -- Degenerate Dimensions
    order_id STRING,
    order_item_id INT,
    -- Measures & Metrics
    price DOUBLE,
    freight_value DOUBLE,
    total_amount DOUBLE,
    delivery_days INT,
    is_late_delivery BOOLEAN,
    -- Metadata
    order_purchase_timestamp TIMESTAMP
) USING DELTA;
""")

# 2. LOAD SOURCE TABLES
df_orders = spark.table("globalmart.bronze.bronze_orders").filter(F.col("order_status") == "delivered")
df_items = spark.table("globalmart.bronze.bronze_order_items")
dim_customer = spark.table("globalmart.silver.dim_customer")
dim_seller = spark.table("globalmart.silver.dim_seller")
dim_product = spark.table("globalmart.silver.dim_product")

# 3. JOIN DELIVERED ORDERS WITH ORDER ITEMS & DIMENSIONS
fct_sales_df = (
    df_items
    .join(df_orders, on="order_id", how="inner")
    # Join Dim Customer on natural key & point-in-time window for SCD Type 2
    .join(
        dim_customer,
        (df_orders.customer_id == dim_customer.customer_id) &
        (df_orders.order_purchase_timestamp >= dim_customer.effective_start_date) &
        (
            dim_customer.effective_end_date.isNull() | 
            (df_orders.order_purchase_timestamp < dim_customer.effective_end_date)
        ),
        how="inner"
    )
    # Join Dim Seller
    .join(dim_seller, on="seller_id", how="inner")
    # Join Dim Product
    .join(dim_product, on="product_id", how="inner")
    # Compute Measures & Foreign Keys
    .withColumn("order_date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
    .withColumn("total_amount", F.round(F.col("price") + F.col("freight_value"), 2))
    .withColumn(
        "delivery_days", 
        F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_purchase_timestamp"))
    )
    .withColumn(
        "is_late_delivery", 
        F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), True).otherwise(False)
    )
    # Generate Fact Surrogate Key
    .withColumn(
        "sales_fact_sk", 
        F.sha2(F.concat_ws("||", F.col("order_id"), F.col("order_item_id").cast("string")), 256)
    )
    .select(
        "sales_fact_sk",
        "customer_sk",
        "seller_sk",
        "product_sk",
        "order_date_key",
        "order_id",
        F.col("order_item_id").cast("int"),
        "price",
        "freight_value",
        "total_amount",
        "delivery_days",
        "is_late_delivery",
        "order_purchase_timestamp"
    )
)

# Overwrite fact table
fct_sales_df.write.format("delta").mode("overwrite").saveAsTable("globalmart.silver.fct_sales")

print(f"✅ `fct_sales` created and populated with {spark.table('globalmart.silver.fct_sales').count():,} delivered order items.")

In [0]:
%sql
-- ==============================================================================
-- VALIDATION 1: ZERO NULLS IN ALL DIMENSION FOREIGN KEYS
-- ==============================================================================
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customer_sk IS NULL THEN 1 ELSE 0 END) AS null_customer_fk,
    SUM(CASE WHEN seller_sk IS NULL THEN 1 ELSE 0 END) AS null_seller_fk,
    SUM(CASE WHEN product_sk IS NULL THEN 1 ELSE 0 END) AS null_product_fk,
    SUM(CASE WHEN order_date_key IS NULL THEN 1 ELSE 0 END) AS null_date_fk
FROM globalmart.silver.fct_sales;

In [0]:
%sql
-- ==============================================================================
-- VALIDATION 2: FACT ROW COUNT VS DELIVERED ORDER ITEMS IN SOURCE
-- ==============================================================================
WITH source_delivered_items AS (
    SELECT COUNT(i.order_id) AS expected_count
    FROM globalmart.bronze.bronze_order_items i
    INNER JOIN globalmart.bronze.bronze_orders o ON i.order_id = o.order_id
    WHERE o.order_status = 'delivered'
),
fact_items AS (
    SELECT COUNT(*) AS actual_count
    FROM globalmart.silver.fct_sales
)
SELECT 
    s.expected_count,
    f.actual_count,
    (f.actual_count - s.expected_count) AS difference
FROM source_delivered_items s
CROSS JOIN fact_items f;

In [0]:
%sql
-- ==============================================================================
-- VALIDATION 3: REVENUE COMPARISON (FACT VS SOURCE DELIVERED ORDERS)
-- ==============================================================================
WITH source_delivered_revenue AS (
    SELECT 
        ROUND(SUM(i.price), 2) AS source_price,
        ROUND(SUM(i.freight_value), 2) AS source_freight,
        ROUND(SUM(i.price + i.freight_value), 2) AS source_total_revenue
    FROM globalmart.bronze.bronze_order_items i
    INNER JOIN globalmart.bronze.bronze_orders o ON i.order_id = o.order_id
    WHERE o.order_status = 'delivered'
),
fact_revenue AS (
    SELECT 
        ROUND(SUM(price), 2) AS fact_price,
        ROUND(SUM(freight_value), 2) AS fact_freight,
        ROUND(SUM(total_amount), 2) AS fact_total_revenue
    FROM globalmart.silver.fct_sales
)
SELECT 
    s.source_price,
    f.fact_price,
    s.source_freight,
    f.fact_freight,
    s.source_total_revenue,
    f.fact_total_revenue,
    ROUND(f.fact_total_revenue - s.source_total_revenue, 2) AS revenue_difference
FROM source_delivered_revenue s
CROSS JOIN fact_revenue f;

In [0]:
%sql
-- ==============================================================================
-- TASK 7.7: STAR SCHEMA ANALYTICAL QUERY
-- ==============================================================================

SELECT 
    d.year_month AS time_period,
    c.customer_state,
    p.product_category_name AS product_category,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(f.sales_fact_sk) AS total_items_sold,
    ROUND(SUM(f.total_amount), 2) AS total_revenue,
    ROUND(AVG(f.delivery_days), 1) AS avg_delivery_days,
    ROUND(
        (SUM(CASE WHEN f.is_late_delivery THEN 1 ELSE 0 END) * 100.0) / COUNT(f.sales_fact_sk), 
        2
    ) AS late_delivery_pct
FROM globalmart.silver.fct_sales f
-- Join Dimension Tables
INNER JOIN globalmart.silver.dim_customer c 
    ON f.customer_sk = c.customer_sk
INNER JOIN globalmart.silver.dim_seller s 
    ON f.seller_sk = s.seller_sk
INNER JOIN globalmart.silver.dim_product p 
    ON f.product_sk = p.product_sk
INNER JOIN globalmart.silver.dim_date d 
    ON f.order_date_key = d.date_key
-- Filters: Year 2018, Selected States (SP, RJ, MG), Current Customer Versions Only
WHERE d.year = 2018
  AND c.customer_state IN ('SP', 'RJ', 'MG')
  AND c.is_current = TRUE
-- Grouping & Aggregations
GROUP BY 
    d.year_month,
    c.customer_state,
    p.product_category_name
ORDER BY 
    total_revenue DESC
LIMIT 20;